# Real-Speech Serving Comparison Lab

This notebook keeps the execution-first style of the original paged dual-cache notebook while making the production comparison compact and repeatable. Every row uses the same Whisper checkpoint, LibriSpeech subset, arrival schedule, token limit, and parity criteria.

### Systems compared

1. **Sequential** — one request at a time.
2. **Static batching** — fixed eager batches.
3. **Static graph** — fixed batches with CUDA-graph decode.
4. **Paged KV engine** — continuous batching with cross-KV gathered during decode.
5. **Cross-KV engine** — continuous batching with cross-KV gathered once and reused.
6. **Graph engine** — persistent CUDA-graph decoder buffers.
7. **Encoder-optimized graph engine** — batched encoder and direct encoder-KV-to-graph-slot handoff.

Correctness stays beside performance: WER and exact-token match are displayed in every main comparison table.


---
## 1. Environment Setup & Experiment Controls

`full` runs the configured real-speech experiment. `quick` uses a smaller sample and one measured repeat. `cached` only renders the existing JSON. Launch Jupyter from the repository root.


In [19]:
%load_ext autoreload
%autoreload 2

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
RUNNER = REPO_ROOT / "scripts" / "run_hybrid_before_after_burst.py"
OUTPUT_JSON = REPO_ROOT / "data" / "streamlined_serving_comparison.json"
if not RUNNER.exists():
    raise RuntimeError("Launch Jupyter from the repository root")

RUN_MODE = os.environ.get("REAL_SPEECH_RUN_MODE", "full")  # full | quick | cached
REPEATS = 3
NUM_REQUESTS = 100
INCLUDE_POISSON = True
SYSTEMS_TO_RUN = [
    "sequential",
    "static_batching",
    "static_graph",
    "hybrid_before",
    "hybrid_cross_kv",
    "hybrid_after",
    "hybrid_encoder_optimized",
]

if RUN_MODE == "quick":
    REPEATS, NUM_REQUESTS = 1, 20
assert RUN_MODE in {"full", "quick", "cached"}
print(f"Mode={RUN_MODE} | requests={NUM_REQUESTS} | repeats={REPEATS}")
print("Systems:", ", ".join(SYSTEMS_TO_RUN))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Mode=full | requests=100 | repeats=3
Systems: sequential, static_batching, static_graph, hybrid_before, hybrid_cross_kv, hybrid_after, hybrid_encoder_optimized


In [20]:
import torch
import transformers

try:
    import triton
    triton_version = triton.__version__
except ImportError:
    triton_version = "not installed"

environment = {
    "Python": sys.version.split()[0],
    "CUDA available": torch.cuda.is_available(),
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "PyTorch": torch.__version__,
    "Transformers": transformers.__version__,
    "Triton": triton_version,
}
display(pd.DataFrame(environment.items(), columns=["Component", "Value"]))


,Component,Value
0,Python,3.10.11
1,CUDA available,True
2,GPU,NVIDIA GeForce RTX 3050 6GB Laptop GPU
3,PyTorch,2.6.0+cu124
4,Transformers,4.57.0
5,Triton,3.8.0


---
## 2. Confirm the Executable Configurations

The table comes directly from the runner. This prevents notebook labels from drifting away from the actual flags used by the engine.


In [21]:
from scripts.run_hybrid_before_after_burst import SYSTEMS

system_configs = {name: config for name, config in SYSTEMS}
config_rows = []
for name in SYSTEMS_TO_RUN:
    config = system_configs[name]
    config_rows.append({
        "System": name,
        "Kind": config["kind"],
        "Decoder": config["decode"],
        "Cross-KV reuse": config.get("cache_cross_kv", False),
        "CUDA graph": config.get("graph_decode", False),
        "Encoder batch": config.get("batch_encoder", False),
        "Direct encoder KV": config.get("graph_direct_encoder_kv", False),
        "Readback K": config.get("readback_k", 1),
    })
display(pd.DataFrame(config_rows).style.hide(axis="index"))


System,Kind,Decoder,Cross-KV reuse,CUDA graph,Encoder batch,Direct encoder KV,Readback K
sequential,sequential,eager,False,False,False,False,1
static_batching,static,eager,False,False,False,False,1
static_graph,static,cuda_graph,False,True,False,False,4
hybrid_before,engine,eager,False,False,False,False,1
hybrid_cross_kv,engine,eager,True,False,False,False,1
hybrid_after,engine,cuda_graph,True,True,False,False,4
hybrid_encoder_optimized,engine,cuda_graph,True,True,True,True,4


---
## 3. Run the Parity-Gated Comparison

The runner performs a correctness preflight, discarded warm-up, cyclic execution ordering, GPU/CPU telemetry, and a sequential canary. A failed telemetry gate does not erase results, but it prevents treating them as clean performance conclusions.


In [22]:
if RUN_MODE != "cached":
    command = [
        sys.executable, "-u", str(RUNNER),
        "--repeats", str(REPEATS),
        "--num-requests", str(NUM_REQUESTS),
        "--systems", ",".join(SYSTEMS_TO_RUN),
        "--output", str(OUTPUT_JSON),
    ]
    if INCLUDE_POISSON:
        command.append("--include-poisson-high")
    subprocess.run(command, cwd=REPO_ROOT, check=True)
elif not OUTPUT_JSON.exists():
    raise FileNotFoundError(f"Cached result not found: {OUTPUT_JSON}")

results = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
print("Session:", results["session_id"])
print("Status:", results["status"])


Session: burst_20261016_234456_528d86
Status: TELEMETRY_INVALID


---
## 4. Correctness Gate

Performance rows are valid only when exact token match is at least the runner threshold and WER remains within its allowed distance from the Hugging Face reference.


In [23]:
parity = pd.DataFrame([
    {
        "System": name,
        "Status": gate["status"],
        "Exact match (%)": gate["exact_match_pct"],
        "WER (%)": gate["wer_pct"],
        "HF WER (%)": gate["reference_wer_pct"],
        "WER delta (points)": gate["wer_delta_points"],
    }
    for name, gate in results["parity_gate"].items()
])
display(parity.round(3).style.hide(axis="index"))


System,Status,Exact match (%),WER (%),HF WER (%),WER delta (points)
sequential,VALID,100.000000,5.282000,5.282000,0.000000
static_batching,VALID,100.000000,5.282000,5.282000,0.000000
static_graph,VALID,100.000000,5.282000,5.282000,0.000000
hybrid_before,VALID,99.000000,5.282000,5.282000,0.000000
hybrid_cross_kv,VALID,99.000000,5.282000,5.282000,0.000000
hybrid_after,VALID,99.000000,5.282000,5.282000,0.000000
hybrid_encoder_optimized,VALID,99.000000,5.282000,5.282000,0.000000


---
## 5. Real-Speech Summary

This is the primary comparison. It follows the original notebook’s wide-table behavior: latency, TTFT, memory, WER, exact match, and throughput are visible together instead of being split across unrelated outputs.


In [24]:
summary_rows = []
for profile_name, profile in results["profiles"].items():
    for system_name, metrics in profile["systems"].items():
        summary_rows.append({
            "Traffic": profile_name,
            "System": system_name,
            "Status": metrics["status"],
            "Capacity (req/s)": metrics["service_capacity_req_per_sec_median"],
            "Audio throughput (x)": metrics["throughput_audio_per_sec_median"],
            "Wall (s)": metrics["wall_time_s_median"],
            "TTFT p50 (s)": metrics["ttft_p50_s_median"],
            "TTFT p95 (s)": metrics["ttft_p95_s_median"],
            "Latency p50 (s)": metrics["latency_p50_s_median"],
            "Latency p95 (s)": metrics["latency_p95_s_median"],
            "Peak VRAM (MB)": metrics["peak_vram_allocated_mb_median"],
            "WER (%)": metrics["wer_pct"],
            "Exact match (%)": metrics["exact_match_pct"],
        })

summary = pd.DataFrame(summary_rows)
for profile_name in results["profiles"]:
    print("=" * 120)
    print(profile_name.upper())
    display(
        summary[summary["Traffic"] == profile_name]
        .drop(columns="Traffic")
        .reset_index(drop=True)
        .round(3)
        .style.hide(axis="index")
        .background_gradient(subset=["Capacity (req/s)"], cmap="Greens")
        .background_gradient(subset=["Latency p95 (s)"], cmap="Reds")
    )


BURST


System,Status,Capacity (req/s),Audio throughput (x),Wall (s),TTFT p50 (s),TTFT p95 (s),Latency p50 (s),Latency p95 (s),Peak VRAM (MB),WER (%),Exact match (%)
sequential,VALID,4.300000,37.914000,23.255000,11.539000,22.239000,11.802000,22.288000,286.960000,5.282000,100.000000
static_batching,VALID,8.854000,78.067000,11.294000,5.589000,10.678000,5.974000,10.884000,385.487000,5.282000,100.000000
static_graph,VALID,17.607000,155.239000,5.679000,3.296000,5.429000,3.423000,5.500000,592.076000,5.282000,100.000000
hybrid_before,VALID,4.177000,36.831000,23.938000,11.689000,22.464000,12.523000,22.973000,1842.810000,5.282000,99.000000
hybrid_cross_kv,VALID,4.869000,42.926000,20.539000,10.040000,19.264000,10.720000,19.748000,1842.810000,5.282000,99.000000
hybrid_after,VALID,11.238000,99.080000,8.899000,4.487000,8.424000,4.734000,8.662000,1870.308000,5.282000,99.000000
hybrid_encoder_optimized,VALID,19.886000,175.327000,5.029000,2.556000,4.730000,2.692000,4.853000,732.886000,5.282000,99.000000


POISSON_HIGH_4_0


System,Status,Capacity (req/s),Audio throughput (x),Wall (s),TTFT p50 (s),TTFT p95 (s),Latency p50 (s),Latency p95 (s),Peak VRAM (MB),WER (%),Exact match (%)
sequential,VALID,4.076000,35.939000,24.532000,1.095000,1.979000,1.310000,2.191000,376.335000,5.282000,100.000000
static_batching,VALID,4.294000,37.857000,23.290000,0.163000,0.410000,0.509000,0.775000,495.888000,5.282000,99.000000
static_graph,VALID,4.364000,38.478000,22.914000,0.037000,0.128000,0.131000,0.239000,586.327000,5.282000,99.000000
hybrid_before,VALID,3.953000,34.849000,25.300000,1.136000,2.122000,2.034000,3.177000,1933.142000,5.282000,99.000000
hybrid_cross_kv,VALID,4.204000,37.069000,23.785000,0.416000,1.287000,1.061000,2.219000,1932.657000,5.282000,99.000000
hybrid_after,VALID,4.348000,38.340000,22.996000,0.078000,0.153000,0.218000,0.424000,1958.694000,5.282000,99.000000
hybrid_encoder_optimized,VALID,4.370000,38.533000,22.881000,0.032000,0.052000,0.124000,0.230000,782.567000,5.282000,99.000000


---
## 6. Streamlined Optimization Comparisons

Ratios are right/left. Capacity above `1.0x` is faster; latency below `1.0x` is lower. These comparisons isolate the main experimental questions without mixing unrelated architectures or decoder modes.


In [25]:
comparison_pairs = [
    ("sequential", "static_batching", "batching vs sequential"),
    ("static_batching", "static_graph", "graph effect on static"),
    ("hybrid_before", "hybrid_cross_kv", "one-time cross-KV reuse"),
    ("hybrid_cross_kv", "hybrid_after", "graph effect on engine"),
    ("hybrid_after", "hybrid_encoder_optimized", "encoder handoff optimization"),
    ("static_graph", "hybrid_encoder_optimized", "matched graph architecture"),
]

ratio_rows = []
for profile_name, profile in results["profiles"].items():
    systems = profile["systems"]
    for left, right, question in comparison_pairs:
        if left not in systems or right not in systems:
            continue
        a, b = systems[left], systems[right]
        ratio_rows.append({
            "Traffic": profile_name,
            "Question": question,
            "Comparison": f"{right} / {left}",
            "Capacity ratio": b["service_capacity_req_per_sec_median"] / a["service_capacity_req_per_sec_median"],
            "Wall ratio": b["wall_time_s_median"] / a["wall_time_s_median"],
            "TTFT p95 ratio": b["ttft_p95_s_median"] / a["ttft_p95_s_median"],
            "Latency p95 ratio": b["latency_p95_s_median"] / a["latency_p95_s_median"],
            "WER delta": b["wer_pct"] - a["wer_pct"],
        })

ratios = pd.DataFrame(ratio_rows)
display(ratios.round(3).style.hide(axis="index"))


Traffic,Question,Comparison,Capacity ratio,Wall ratio,TTFT p95 ratio,Latency p95 ratio,WER delta
burst,batching vs sequential,static_batching / sequential,2.059000,0.486000,0.480000,0.488000,0.000000
burst,graph effect on static,static_graph / static_batching,1.989000,0.503000,0.508000,0.505000,0.000000
burst,one-time cross-KV reuse,hybrid_cross_kv / hybrid_before,1.165000,0.858000,0.858000,0.860000,0.000000
burst,graph effect on engine,hybrid_after / hybrid_cross_kv,2.308000,0.433000,0.437000,0.439000,0.000000
burst,encoder handoff optimization,hybrid_encoder_optimized / hybrid_after,1.770000,0.565000,0.561000,0.560000,0.000000
burst,matched graph architecture,hybrid_encoder_optimized / static_graph,1.129000,0.885000,0.871000,0.882000,0.000000
poisson_high_4_0,batching vs sequential,static_batching / sequential,1.053000,0.949000,0.207000,0.354000,0.000000
poisson_high_4_0,graph effect on static,static_graph / static_batching,1.016000,0.984000,0.312000,0.309000,0.000000
poisson_high_4_0,one-time cross-KV reuse,hybrid_cross_kv / hybrid_before,1.064000,0.940000,0.606000,0.698000,0.000000
poisson_high_4_0,graph effect on engine,hybrid_after / hybrid_cross_kv,1.034000,0.967000,0.119000,0.191000,0.000000


---
## 7. Repeat-Level Results

Medians can hide order and thermal effects. Use these rows to inspect variability before accepting an optimization.


In [26]:
repeat_rows = []
for profile_name, profile in results["profiles"].items():
    for system_name, metrics in profile["systems"].items():
        for run in metrics["runs"]:
            repeat_rows.append({
                "Traffic": profile_name,
                "System": system_name,
                "Repeat": run["repeat"],
                "Order": run["order_index"],
                "Capacity (req/s)": run["service_capacity_req_per_sec"],
                "Latency p95 (s)": run["latency_p95_s"],
                "TTFT p95 (s)": run["ttft_p95_s"],
                "Exact match (%)": run["exact_match_pct"],
                "WER (%)": run["wer_pct"],
                "Run gate": run["run_gate_passed"],
            })
display(pd.DataFrame(repeat_rows).round(3).style.hide(axis="index"))


Traffic,System,Repeat,Order,Capacity (req/s),Latency p95 (s),TTFT p95 (s),Exact match (%),WER (%),Run gate
burst,sequential,1,1,4.337000,22.116000,22.067000,100.000000,5.282000,True
burst,sequential,2,7,4.277000,22.426000,22.375000,100.000000,5.282000,True
burst,sequential,3,6,4.300000,22.288000,22.239000,100.000000,5.282000,True
burst,static_batching,1,2,8.854000,10.884000,10.678000,100.000000,5.282000,True
burst,static_batching,2,1,8.811000,10.920000,10.732000,100.000000,5.282000,True
burst,static_batching,3,7,8.861000,10.867000,10.673000,100.000000,5.282000,True
burst,static_graph,1,3,17.607000,5.500000,5.429000,100.000000,5.282000,True
burst,static_graph,2,2,17.288000,5.601000,5.527000,100.000000,5.282000,False
burst,static_graph,3,1,18.035000,5.364000,5.292000,100.000000,5.282000,False
burst,hybrid_before,1,4,4.177000,22.973000,22.464000,99.000000,5.282000,True


---
## 8. Session Quality Gate

Do not publish speed conclusions unless this gate passes. A parity-valid but telemetry-invalid run remains useful for debugging only.


In [27]:
gate = results["session_gate"]
canary = results["canary"]
gate_rows = [{
    "Session status": results["status"],
    "Gate passed": gate["gate_passed"],
    "Busy throttling (%)": gate["max_busy_throttle_pct"],
    "Clock spread (%)": gate["clock_spread_pct"],
    "Max temperature (C)": gate["max_temperature_c"],
    "Canary drift (%)": canary["drift_pct"],
}]
display(pd.DataFrame(gate_rows).round(3).style.hide(axis="index"))

if gate["failures"]:
    print("Failing criteria:")
    for failure in gate["failures"]:
        print(" -", failure)
else:
    print("All telemetry criteria passed.")


Session status,Gate passed,Busy throttling (%),Clock spread (%),Max temperature (C),Canary drift (%)
TELEMETRY_INVALID,False,100.000000,39.256000,76.000000,0.217000


Failing criteria:
 - Peak busy throttling ratio 100.0% exceeded 10.0%.
 - Cross-system busy median clock spread (39.3%) exceeds 10.0% limit (min=1102 MHz, max=1815 MHz).
